# 项目三：模拟均值回归股票中的强化学习方法比较

**价格动态：**
$$L_{t+1} = (1-\kappa)L_t + \sigma Z_t, \quad Z_t \sim \mathcal{N}(0,1), \quad S_{t+1} = S_t e^{L_{t+1}}$$

**奖励函数 (1)：** $R_t = A_t(S_{t+1}-S_t) - \lambda A_t^2$  
**奖励函数 (2)：** $R_t = A_t(S_{t+1}-S_t) - \lambda(A_t - A_{t-1})^2 S_t$

**目标：** $\max_\pi \mathbb{E}^\pi\left[\sum_{t=0}^\infty \gamma^t R_t\right]$

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal
from copy import deepcopy
from collections import deque
from functools import partial
import random
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

plt.rcParams.update({'figure.dpi': 100})
print(f'PyTorch {torch.__version__}  |  NumPy {np.__version__}')

## 1. 环境

状态变量与转移动态：

| 奖励函数 | 状态 $s_t$ | 维度 |
|--------|------------|-----|
| (1) | $(L_t,\, \log S_t)$ | 2 |
| (2) | $(L_t,\, \log S_t,\, A_{t-1})$ | 3 |

**Oracle 模式：** 可随时调用 `reset()` 开始新的独立回合。  
**序列模式：** 只有一条连续轨迹——`reset()` 仅在初始时调用一次。

In [ ]:
# 对数价格上限：将 log(S) 截断到 [-LOG_S_CLIP, LOG_S_CLIP]（即 S ∈ [exp(-3), exp(3)] ≈ [0.05, 20]）。
# 若不加限制，对数正态股价在长期轨迹中可增长至天文数字，
# 当 γ * exp(σ²/κ) ≥ 1（此处 ≈ 1.094）时，无约束最优动作 A* ∝ S_t，导致期望累积奖励发散。
# 该上限相当于现实市场中的"熔断机制"。
LOG_S_CLIP = 3.0


class MeanRevertingEnv:
    """
    均值回归股票环境。
    无约束动态：L_{t+1} = (1-κ)L_t + σZ_t，S_{t+1} = S_t·exp(L_{t+1})
    加截断后：log(S_t) 限制在 [-LOG_S_CLIP, LOG_S_CLIP]；L_t 存储实际收益率。

    状态（reward_type=1）：[L_t, log(S_t)]           维度=2
    状态（reward_type=2）：[L_t, log(S_t), A_{t-1}]  维度=3
    """
    def __init__(self, kappa=0.1, sigma=0.1, lam=0.1, gamma=0.99,
                 reward_type=1, max_steps=200):
        self.kappa = kappa
        self.sigma = sigma
        self.lam = lam
        self.gamma = gamma
        self.reward_type = reward_type
        self.max_steps = max_steps

    @property
    def state_dim(self):
        return 2 if self.reward_type == 1 else 3

    def reset(self):
        self.L = 0.0
        self.log_S = 0.0   # log(S_0) = 0  →  S_0 = 1
        self.S = 1.0
        self.A_prev = 0.0
        self.t = 0
        return self._state()

    def _state(self):
        s = [self.L, self.log_S]
        if self.reward_type == 2:
            s.append(self.A_prev)
        return np.array(s, dtype=np.float32)

    def step(self, action):
        A = float(action)
        # AR(1) 无约束对数收益率
        L_raw = (1.0 - self.kappa) * self.L + self.sigma * np.random.randn()
        # 截断累积对数价格，防止数值爆炸
        log_S_next = np.clip(self.log_S + L_raw, -LOG_S_CLIP, LOG_S_CLIP)
        S_next = np.exp(log_S_next)
        # 实际收益率（价格触边界时 ≤ L_raw）
        L_next = log_S_next - self.log_S

        if self.reward_type == 1:
            reward = A * (S_next - self.S) - self.lam * A**2
        else:
            reward = A * (S_next - self.S) - self.lam * (A - self.A_prev)**2 * self.S

        self.L = L_next
        self.log_S = log_S_next
        self.S = S_next
        self.A_prev = A
        self.t += 1
        return self._state(), reward, self.t >= self.max_steps


# 默认超参数
KAPPA, SIGMA, LAM, GAMMA = 0.1, 0.1, 0.1, 0.99
DEFAULT_PARAMS = (KAPPA, SIGMA, LAM, GAMMA)


def evaluate_policy(policy_fn, kappa=KAPPA, sigma=SIGMA, lam=LAM, gamma=GAMMA,
                    reward_type=1, n_paths=500, T=200, seed=0):
    """蒙特卡洛评估策略。返回 (均值, 标准误)。"""
    np.random.seed(seed)
    env = MeanRevertingEnv(kappa, sigma, lam, gamma, reward_type, max_steps=T)
    disc_rewards = []
    for _ in range(n_paths):
        state = env.reset()
        total = 0.0
        for t in range(T):
            action = policy_fn(state)
            state, reward, done = env.step(action)
            total += (gamma ** t) * reward
            if done:
                break
        disc_rewards.append(total)
    arr = np.array(disc_rewards)
    return float(arr.mean()), float(arr.std() / np.sqrt(len(arr)))

In [ ]:
# 可视化样本轨迹
np.random.seed(0)
env_demo = MeanRevertingEnv(KAPPA, SIGMA, LAM, GAMMA, max_steps=300)
state = env_demo.reset()
L_path, S_path = [0.0], [1.0]
for _ in range(299):
    state, _, done = env_demo.step(0.0)
    L_path.append(env_demo.L)
    S_path.append(env_demo.S)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.plot(L_path, color='steelblue')
ax1.axhline(0, color='r', linestyle='--', alpha=0.5, label='均值')
ax1.set(title='对数收益率 $L_t$（均值回归）', xlabel='时间步 $t$', ylabel='$L_t$')
ax1.legend()

ax2.plot(S_path, color='darkorange')
ax2.axhline(1.0, color='r', linestyle='--', alpha=0.5)
ax2.set(title='股票价格 $S_t$', xlabel='时间步 $t$', ylabel='$S_t$')

plt.tight_layout()
plt.savefig('fig_trajectories.png', bbox_inches='tight')
plt.show()

## 2. 奖励函数 (1) 的解析最优策略

**核心思路：** 在奖励函数 (1) 中，动作 $A_t$ **不影响** 下一步状态的转移 $(S_{t+1}, L_{t+1})$。因此 Bellman 方程可以解耦：

$$V^*(s) = \max_{A_t}\underbrace{\bigl[A_t\,\mathbb{E}[S_{t+1}-S_t|s] - \lambda A_t^2\bigr]}_{\text{对 }A_t\text{ 求最优}} + \gamma\,\mathbb{E}[V^*(s')]$$

一阶条件 $\Rightarrow$ **全局最优**（而非仅仅贪心最优）：

$$A_t^* = \frac{\mathbb{E}[S_{t+1}-S_t|L_t, S_t]}{2\lambda}$$

由于 $L_{t+1}|L_t \sim \mathcal{N}(L_t(1-\kappa),\,\sigma^2)$：

$$\mathbb{E}[S_{t+1}|S_t,L_t] = S_t\exp\!\left(L_t(1-\kappa)+\frac{\sigma^2}{2}\right)$$

$$\boxed{A_t^* = \frac{S_t\left(e^{L_t(1-\kappa)+\sigma^2/2}-1\right)}{2\lambda}}$$

In [ ]:
def optimal_policy_r1(state, kappa=KAPPA, sigma=SIGMA, lam=LAM):
    """奖励函数 (1) 的解析最优策略。"""
    L_t, log_S_t = state[0], state[1]
    S_t = np.exp(log_S_t)
    E_S_next = S_t * np.exp(L_t * (1 - kappa) + 0.5 * sigma**2)
    return (E_S_next - S_t) / (2.0 * lam)


analytical_fn = partial(optimal_policy_r1, kappa=KAPPA, sigma=SIGMA, lam=LAM)
mean_ana, se_ana = evaluate_policy(analytical_fn, reward_type=1, n_paths=1000)
print(f'解析最优策略（奖励函数 1）：{mean_ana:.4f} ± {2*se_ana:.4f}  （95% 置信区间半宽）')

# 在 S_t = 1 处，可视化 A* 关于 L_t 的函数
L_grid = np.linspace(-0.5, 0.5, 300)
A_star = [analytical_fn(np.array([L, 0.0])) for L in L_grid]  # log(S=1)=0

plt.figure(figsize=(7, 3.5))
plt.plot(L_grid, A_star, 'b-', lw=2)
plt.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
plt.axvline(0, color='k', lw=0.8, ls='--', alpha=0.4)
plt.xlabel('$L_t$'); plt.ylabel('$A^*_t$')
plt.title('解析最优动作 vs 对数收益率 $L_t$（$S_t=1$）')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_analytical.png', bbox_inches='tight')
plt.show()

## 3. 神经网络结构

所有网络采用统一模板：**2 个隐藏层 × 64 个神经元**。

| 网络 | 输入 | 输出 | 激活函数 |
|---------|-------|--------|------------|
| PPO Actor（演员） | $s_t$ | $\mu(s)$；全局 $\log\sigma$ | Tanh 隐藏层 |
| PPO Critic（评论家） | $s_t$ | $V(s)$ | Tanh 隐藏层 |
| SAC Actor（演员） | $s_t$ | $(\mu(s),\log\sigma(s))$ | ReLU 隐藏层 |
| SAC Q 网络 | $(s_t, a_t)$ | $Q(s,a)$ | ReLU 隐藏层 |

In [ ]:
# ─────────────────── PPO 网络 ───────────────────
class ActorNet(nn.Module):
    """PPO 演员网络：高斯策略，均值由状态决定，log_std 为全局可学习参数。"""
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.mean_net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),   nn.Tanh(),
            nn.Linear(hidden, 1)
        )
        self.log_std = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mean = self.mean_net(x)
        std  = self.log_std.exp().expand_as(mean)
        return Normal(mean, std)

    def evaluate_actions(self, states, actions):
        dist = self(states)
        return dist.log_prob(actions).sum(-1), dist.entropy().sum(-1)


class CriticNet(nn.Module):
    """PPO 评论家网络：状态价值函数 V(s)。"""
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),   nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ─────────────────── SAC 网络 ───────────────────
ACTION_SCALE = 5.0  # SAC 动作通过 tanh 压缩后限制在 [-5, 5]


class SACActorNet(nn.Module):
    """
    SAC 演员网络，带 tanh 压缩：action = tanh(u) * ACTION_SCALE。
    将动作限制在 [-ACTION_SCALE, ACTION_SCALE]，防止 Q 值自举时数值爆炸。
    对数概率中包含 tanh 变量替换的 Jacobian 修正项。
    """
    LOG_STD_MIN, LOG_STD_MAX = -5.0, 2.0

    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU()
        )
        self.mean_head    = nn.Linear(hidden, 1)
        self.log_std_head = nn.Linear(hidden, 1)

    def _pre_squash(self, x):
        """返回压缩前高斯分布的 (均值, 标准差)。"""
        h = self.shared(x)
        mean    = self.mean_head(h)
        log_std = self.log_std_head(h).clamp(self.LOG_STD_MIN, self.LOG_STD_MAX)
        return mean, log_std.exp()

    def sample(self, x):
        """重参数化采样 + tanh 压缩 + 修正对数概率。"""
        mean, std = self._pre_squash(x)
        dist = Normal(mean, std)
        u    = dist.rsample()
        action = torch.tanh(u) * ACTION_SCALE
        # tanh 变换的 Jacobian 修正
        log_p = dist.log_prob(u) - torch.log(
            ACTION_SCALE * (1 - (action / ACTION_SCALE).pow(2)) + 1e-6
        )
        return action, log_p.sum(-1, keepdim=True)

    def get_action(self, state_np, deterministic=False):
        s = torch.FloatTensor(state_np).unsqueeze(0)
        with torch.no_grad():
            mean, _ = self._pre_squash(s)
            if deterministic:
                return (torch.tanh(mean) * ACTION_SCALE).item()
            else:
                action, _ = self.sample(s)
                return action.item()


class QNet(nn.Module):
    """SAC 评论家网络：Q(s, a)。"""
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + 1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),        nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=-1))


print(f'网络类定义完成。ACTION_SCALE={ACTION_SCALE}')

## 4. PPO（同策略）

**算法流程：**
1. 在当前策略 $\pi_\theta$ 下收集长度为 $T_{\text{roll}}$ 的轨迹片段。
2. 使用 **GAE**（$\lambda=0.95$）计算优势估计：$\hat{A}_t = \sum_{k\ge0}(\gamma\lambda)^k\delta_{t+k}$，其中 $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$。
3. 进行 $K$ 轮 mini-batch 更新：
$$\mathcal{L}^{\text{CLIP}} = -\mathbb{E}\bigl[\min(r_t\hat{A}_t,\;\text{clip}(r_t,1\pm\varepsilon)\hat{A}_t)\bigr] - c_H H[\pi]$$
$$\mathcal{L}^{\text{VF}} = \mathbb{E}\bigl[(V(s_t)-\hat{R}_t)^2\bigr]$$
其中 $r_t = \pi_\theta(a_t|s_t)/\pi_{\theta_{\text{old}}}(a_t|s_t)$ 为重要性采样比率。

**Oracle 模式：** 每次采集完后重置环境。  
**序列模式：** 不重置——保持一条连续轨迹。

In [ ]:
def compute_gae(rewards, values, dones, next_value, gamma, lam_gae=0.95):
    """广义优势估计（GAE）。"""
    n   = len(rewards)
    adv = np.zeros(n, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(n)):
        nv    = values[t + 1] if t + 1 < n else next_value
        mask  = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * nv * mask - values[t]
        gae   = delta + gamma * lam_gae * mask * gae
        adv[t] = gae
    return adv, adv + np.array(values, dtype=np.float32)


def train_ppo(kappa=KAPPA, sigma=SIGMA, lam=LAM, gamma=GAMMA,
              reward_type=1, oracle=True, n_steps=120_000,
              rollout_len=2048, lr=3e-4, gae_lam=0.95,
              clip_eps=0.2, n_epochs=10, mini_batch=256,
              ent_coef=0.01, hidden=64, verbose=True):
    """训练 PPO。返回 (actor, (步数日志, 回报日志))。"""
    env    = MeanRevertingEnv(kappa, sigma, lam, gamma, reward_type, max_steps=200)
    actor  = ActorNet(env.state_dim, hidden)
    critic = CriticNet(env.state_dim, hidden)
    opt    = torch.optim.Adam(
        list(actor.parameters()) + list(critic.parameters()), lr=lr
    )

    state      = env.reset()
    ep_reward  = 0.0
    recent     = deque(maxlen=20)
    log_steps, log_ret = [], []
    total_steps = 0

    while total_steps < n_steps:
        s_buf, a_buf, lp_buf, r_buf, d_buf, v_buf = [], [], [], [], [], []

        for _ in range(rollout_len):
            st = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                dist = actor(st)
                at   = dist.sample()
                lpt  = dist.log_prob(at).sum(-1)
                vt   = critic(st)

            next_state, reward, done = env.step(at.item())
            s_buf.append(state.copy())
            a_buf.append([at.item()])
            lp_buf.append(lpt.item())
            r_buf.append(reward)
            d_buf.append(done)
            v_buf.append(vt.item())

            ep_reward += reward
            state = next_state
            if done:
                recent.append(ep_reward)
                ep_reward = 0.0
                if oracle:
                    state = env.reset()

        # 自举最终状态价值
        with torch.no_grad():
            nv = critic(torch.FloatTensor(state).unsqueeze(0)).item()

        adv, ret = compute_gae(r_buf, v_buf, d_buf, nv, gamma, gae_lam)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        s_t   = torch.FloatTensor(np.array(s_buf))
        a_t   = torch.FloatTensor(a_buf)
        old_lp = torch.FloatTensor(lp_buf)
        adv_t  = torch.FloatTensor(adv)
        ret_t  = torch.FloatTensor(ret)

        for _ in range(n_epochs):
            perm = np.random.permutation(rollout_len)
            for start in range(0, rollout_len, mini_batch):
                idx    = perm[start:start + mini_batch]
                new_lp, ent = actor.evaluate_actions(s_t[idx], a_t[idx])
                ratio  = (new_lp - old_lp[idx]).exp()
                surr   = torch.min(
                    ratio * adv_t[idx],
                    ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv_t[idx]
                )
                a_loss = -surr.mean() - ent_coef * ent.mean()
                c_loss = (critic(s_t[idx]) - ret_t[idx]).pow(2).mean()

                opt.zero_grad()
                (a_loss + 0.5 * c_loss).backward()
                nn.utils.clip_grad_norm_(
                    list(actor.parameters()) + list(critic.parameters()), 0.5)
                opt.step()

        total_steps += rollout_len
        if recent:
            log_steps.append(total_steps)
            log_ret.append(float(np.mean(recent)))
        if verbose and total_steps % 24576 < rollout_len:
            print(f'  [PPO 奖励{reward_type} {"oracle" if oracle else "序列"}]'
                  f'  步数 {total_steps:>7d}  平均回报 {log_ret[-1]:+.3f}')

    return actor, (log_steps, log_ret)


print('PPO 定义完成。')

## 5. SAC（异策略）

**算法流程：**
1. 将转移 $(s,a,r,s')$ 存入经验回放池 $\mathcal{D}$（容量 100K）。
2. 采样 mini-batch，计算软 Bellman 目标：
$$y = r + \gamma\bigl(\min_{i=1,2}Q_{\phi_i^\text{tgt}}(s',\tilde{a}') - \alpha\log\pi_\theta(\tilde{a}'|s')\bigr)$$
3. 更新双 Q 网络：$\min_\phi\mathbb{E}[(Q_\phi(s,a)-y)^2]$。
4. 更新演员：$\min_\theta\mathbb{E}[\alpha\log\pi_\theta(\tilde{a}|s) - \min_i Q_{\phi_i}(s,\tilde{a})]$。
5. 自动调节温度参数 $\alpha$：$\min_\alpha \mathbb{E}[-\alpha(\log\pi + H_\text{target})]$。
6. 软更新目标网络：$\phi^\text{tgt}\leftarrow \tau\phi+(1-\tau)\phi^\text{tgt}$。

**Oracle 模式：** 当 `done=True` 时重置环境。  
**序列模式：** 经验回放池由单条永不中断的轨迹填充。

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, s_, done):
        self.buf.append((s.copy(), [float(a)], [float(r)], s_.copy(), [float(done)]))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s_, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)),
                torch.FloatTensor(np.array(a)),
                torch.FloatTensor(np.array(r)),
                torch.FloatTensor(np.array(s_)),
                torch.FloatTensor(np.array(d)))

    def __len__(self):
        return len(self.buf)


def soft_update(net, target, tau=0.005):
    for p, pt in zip(net.parameters(), target.parameters()):
        pt.data.copy_(tau * p.data + (1.0 - tau) * pt.data)


def train_sac(kappa=KAPPA, sigma=SIGMA, lam=LAM, gamma=GAMMA,
              reward_type=1, oracle=True, n_steps=120_000,
              lr=3e-4, batch_size=256, tau=0.005,
              init_alpha=0.2, warmup=2000, hidden=64, verbose=True):
    """训练 SAC。返回 (actor, (步数日志, 回报日志))。"""
    env   = MeanRevertingEnv(kappa, sigma, lam, gamma, reward_type, max_steps=200)
    actor = SACActorNet(env.state_dim, hidden)
    q1, q2           = QNet(env.state_dim, hidden), QNet(env.state_dim, hidden)
    q1_tgt, q2_tgt   = deepcopy(q1), deepcopy(q2)

    actor_opt = torch.optim.Adam(actor.parameters(), lr=lr)
    q_opt     = torch.optim.Adam(
        list(q1.parameters()) + list(q2.parameters()), lr=lr)

    log_alpha = torch.tensor(np.log(init_alpha), requires_grad=True)
    alpha_opt = torch.optim.Adam([log_alpha], lr=lr)
    target_H  = -1.0  # 1 维动作的目标熵

    buffer    = ReplayBuffer()
    state     = env.reset()
    ep_reward = 0.0
    recent    = deque(maxlen=20)
    log_steps, log_ret = [], []

    for step in range(n_steps):
        if step < warmup:
            action = np.random.randn() * 0.05
        else:
            action = actor.get_action(state, deterministic=False)

        next_state, reward, done = env.step(action)
        # 截断单步奖励，稳定 Q 值目标
        reward_clipped = float(np.clip(reward, -50.0, 50.0))
        buffer.push(state, action, reward_clipped, next_state, done)
        state      = next_state
        ep_reward += reward  # 记录原始奖励用于日志

        if done:
            recent.append(ep_reward)
            ep_reward = 0.0
            if oracle:
                state = env.reset()

        if len(buffer) >= batch_size:
            s, a, r, s_, d = buffer.sample(batch_size)
            alpha = log_alpha.exp().detach()

            with torch.no_grad():
                a_next, lp_next = actor.sample(s_)
                q_next = torch.min(q1_tgt(s_, a_next),
                                   q2_tgt(s_, a_next)) - alpha * lp_next
                y = r + gamma * (1.0 - d) * q_next

            q_loss = (q1(s, a) - y).pow(2).mean() + (q2(s, a) - y).pow(2).mean()
            q_opt.zero_grad()
            q_loss.backward()
            nn.utils.clip_grad_norm_(list(q1.parameters()) + list(q2.parameters()), 1.0)
            q_opt.step()

            a_new, lp_new = actor.sample(s)
            q_min         = torch.min(q1(s, a_new), q2(s, a_new))
            actor_loss    = (alpha * lp_new - q_min).mean()
            actor_opt.zero_grad()
            actor_loss.backward()
            nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
            actor_opt.step()

            alpha_loss = -(log_alpha * (lp_new.detach() + target_H)).mean()
            alpha_opt.zero_grad(); alpha_loss.backward(); alpha_opt.step()

            soft_update(q1, q1_tgt, tau)
            soft_update(q2, q2_tgt, tau)

        if (step + 1) % 2000 == 0 and recent:
            log_steps.append(step + 1)
            log_ret.append(float(np.mean(recent)))
        if verbose and (step + 1) % 24000 == 0 and recent:
            print(f'  [SAC 奖励{reward_type} {"oracle" if oracle else "序列"}]'
                  f'  步数 {step+1:>7d}  平均回报 {log_ret[-1]:+.3f}')

    return actor, (log_steps, log_ret)


print('SAC 定义完成。')

## 6. 训练——奖励函数 (1)

四种实验条件：PPO / SAC × Oracle 模式 / 序列模式。

In [ ]:
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

print('=== 奖励函数 (1) ===')
ppo_r1_oracle, ppo_r1_oracle_log = train_ppo(reward_type=1, oracle=True)
ppo_r1_seq,    ppo_r1_seq_log    = train_ppo(reward_type=1, oracle=False)
sac_r1_oracle, sac_r1_oracle_log = train_sac(reward_type=1, oracle=True)
sac_r1_seq,    sac_r1_seq_log    = train_sac(reward_type=1, oracle=False)
print('训练完成。')

## 7. 训练——奖励函数 (2)

In [ ]:
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

print('=== 奖励函数 (2) ===')
ppo_r2_oracle, ppo_r2_oracle_log = train_ppo(reward_type=2, oracle=True)
ppo_r2_seq,    ppo_r2_seq_log    = train_ppo(reward_type=2, oracle=False)
sac_r2_oracle, sac_r2_oracle_log = train_sac(reward_type=2, oracle=True)
sac_r2_seq,    sac_r2_seq_log    = train_sac(reward_type=2, oracle=False)
print('训练完成。')

## 8. 评估结果

所有策略均通过**蒙特卡洛**方法评估（500 条独立路径，每条长度 $T=200$）。

In [ ]:
def ppo_policy(actor, deterministic=True):
    def fn(state):
        s = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            dist = actor(s)
            return (dist.mean if deterministic else dist.sample()).item()
    return fn

def sac_policy(actor):
    return lambda state: actor.get_action(state, deterministic=True)


rows = []

# 奖励函数 (1)
m, e = evaluate_policy(analytical_fn, reward_type=1)
rows.append({'方法': '解析最优策略', '奖励函数': 1, '模式': '—', '均值': m, '±2SE': 2*e})

for label, actor, rtype, setting in [
    ('PPO', ppo_r1_oracle, 1, 'Oracle'),
    ('PPO', ppo_r1_seq,    1, '序列'),
    ('SAC', sac_r1_oracle, 1, 'Oracle'),
    ('SAC', sac_r1_seq,    1, '序列'),
    ('PPO', ppo_r2_oracle, 2, 'Oracle'),
    ('PPO', ppo_r2_seq,    2, '序列'),
    ('SAC', sac_r2_oracle, 2, 'Oracle'),
    ('SAC', sac_r2_seq,    2, '序列'),
]:
    if label == 'PPO':
        fn = ppo_policy(actor)
    else:
        fn = sac_policy(actor)
    m, e = evaluate_policy(fn, reward_type=rtype)
    rows.append({'方法': label, '奖励函数': rtype, '模式': setting,
                 '均值': round(m, 4), '±2SE': round(2*e, 4)})

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (oracle_log, seq_log, title) in zip(axes, [
    (ppo_r1_oracle_log, ppo_r1_seq_log, 'PPO — 奖励函数 (1)'),
    (sac_r1_oracle_log, sac_r1_seq_log, 'SAC — 奖励函数 (1)'),
]):
    ax.plot(oracle_log[0], oracle_log[1], label='Oracle 模式')
    ax.plot(seq_log[0],    seq_log[1],    label='序列模式')
    ax.axhline(mean_ana, color='k', ls='--', lw=1.2, label=f'解析最优 ({mean_ana:.3f})')
    ax.set(title=title, xlabel='环境步数', ylabel='平均回合回报')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_training_curves.png', bbox_inches='tight')
plt.show()

## 9. 策略可视化

将学到的动作 $A_t$ 绘制为状态的函数。  
对于**奖励函数 (1)**，与解析最优策略 $A_t^*$ 进行对比。

In [ ]:
# ── 奖励函数 (1)：A(L_t, log S_t) 热力图 ──
L_vals  = np.linspace(-0.4, 0.4, 60)
lS_vals = np.linspace(-0.5, 0.5, 60)
LL, SS  = np.meshgrid(L_vals, lS_vals)

def policy_grid(policy_fn, LL, SS):
    A = np.zeros_like(LL)
    for i in range(LL.shape[0]):
        for j in range(LL.shape[1]):
            A[i, j] = policy_fn(np.array([LL[i,j], SS[i,j]], dtype=np.float32))
    return A

A_ana  = policy_grid(analytical_fn,                 LL, SS)
A_ppo  = policy_grid(ppo_policy(ppo_r1_oracle),     LL, SS)
A_sac  = policy_grid(sac_policy(sac_r1_oracle),     LL, SS)

vmax = max(np.abs(A_ana).max(), np.abs(A_ppo).max(), np.abs(A_sac).max())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, A, title in zip(axes,
    [A_ana, A_ppo, A_sac],
    ['解析最优 $A^*$', 'PPO（Oracle 模式）', 'SAC（Oracle 模式）']):
    im = ax.pcolormesh(L_vals, lS_vals, A,
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
    fig.colorbar(im, ax=ax, label='$A_t$')
    ax.set(xlabel='$L_t$', ylabel='$\\log S_t$', title=title)
    ax.axvline(0, color='k', lw=0.7, ls='--'); ax.axhline(0, color='k', lw=0.7, ls='--')

plt.suptitle('奖励函数 (1) — 策略热力图  $(A_t$ vs $L_t,\\log S_t)$', y=1.02)
plt.tight_layout()
plt.savefig('fig_policy_heatmap_r1.png', bbox_inches='tight')
plt.show()

# ── 奖励函数 (1)：在 log(S)=0 处的一维切片对比 ──
L_fine = np.linspace(-0.5, 0.5, 200)
states_1d = [np.array([L, 0.0], dtype=np.float32) for L in L_fine]

plt.figure(figsize=(8, 4))
plt.plot(L_fine, [analytical_fn(s) for s in states_1d],              'k-',  lw=2,   label='解析最优')
plt.plot(L_fine, [ppo_policy(ppo_r1_oracle)(s) for s in states_1d],  'b--', lw=1.5, label='PPO Oracle')
plt.plot(L_fine, [ppo_policy(ppo_r1_seq)(s)    for s in states_1d],  'b:',  lw=1.5, label='PPO 序列')
plt.plot(L_fine, [sac_policy(sac_r1_oracle)(s) for s in states_1d],  'r--', lw=1.5, label='SAC Oracle')
plt.plot(L_fine, [sac_policy(sac_r1_seq)(s)    for s in states_1d],  'r:',  lw=1.5, label='SAC 序列')
plt.axhline(0, color='k', lw=0.5, ls='--'); plt.axvline(0, color='k', lw=0.5, ls='--')
plt.xlabel('$L_t$'); plt.ylabel('$A_t$')
plt.title('奖励函数 (1) — 在 $S_t=1$ 处的策略切片')
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_policy_slice_r1.png', bbox_inches='tight')
plt.show()

# ── 奖励函数 (2)：在 log(S)=0 处，(L_t, A_{t-1}) 热力图 ──
L_r2   = np.linspace(-0.4, 0.4, 50)
Ap_r2  = np.linspace(-2.0, 2.0, 50)
LL2, AP2 = np.meshgrid(L_r2, Ap_r2)

def policy_grid_r2(policy_fn):
    A = np.zeros_like(LL2)
    for i in range(LL2.shape[0]):
        for j in range(LL2.shape[1]):
            s = np.array([LL2[i,j], 0.0, AP2[i,j]], dtype=np.float32)
            A[i, j] = policy_fn(s)
    return A

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, A, title in zip(axes,
    [policy_grid_r2(ppo_policy(ppo_r2_oracle)),
     policy_grid_r2(sac_policy(sac_r2_oracle))],
    ['PPO Oracle 模式', 'SAC Oracle 模式']):
    vmax2 = np.abs(A).max()
    im = ax.pcolormesh(L_r2, Ap_r2, A,
                       cmap='RdBu_r', vmin=-vmax2, vmax=vmax2, shading='auto')
    fig.colorbar(im, ax=ax, label='$A_t$')
    ax.set(xlabel='$L_t$', ylabel='$A_{t-1}$', title=f'奖励函数 (2) — {title}')

plt.tight_layout()
plt.savefig('fig_policy_heatmap_r2.png', bbox_inches='tight')
plt.show()

## 10. 参数敏感性分析

每次只改变一个参数 $(\kappa, \sigma, \lambda, \gamma)$，在奖励函数 (1) 上重新训练 PPO Oracle，与解析最优策略对比。

In [ ]:
param_sets = [
    {'label': '基准参数',          'kappa': 0.1, 'sigma': 0.1, 'lam': 0.1, 'gamma': 0.99},
    {'label': '快速回归 κ=0.5',    'kappa': 0.5, 'sigma': 0.1, 'lam': 0.1, 'gamma': 0.99},
    {'label': '高波动 σ=0.3',      'kappa': 0.1, 'sigma': 0.3, 'lam': 0.1, 'gamma': 0.99},
    {'label': '高惩罚 λ=1.0',      'kappa': 0.1, 'sigma': 0.1, 'lam': 1.0, 'gamma': 0.99},
    {'label': '低折扣 γ=0.95',     'kappa': 0.1, 'sigma': 0.1, 'lam': 0.1, 'gamma': 0.95},
]

sens_rows = []
for ps in param_sets:
    print(f"训练中：{ps['label']}")
    actor_ps, _ = train_ppo(
        kappa=ps['kappa'], sigma=ps['sigma'], lam=ps['lam'], gamma=ps['gamma'],
        reward_type=1, oracle=True, n_steps=80_000, verbose=False
    )
    ana_fn = partial(optimal_policy_r1,
                     kappa=ps['kappa'], sigma=ps['sigma'], lam=ps['lam'])
    m_rl,  e_rl  = evaluate_policy(ppo_policy(actor_ps),
                                   kappa=ps['kappa'], sigma=ps['sigma'],
                                   lam=ps['lam'], gamma=ps['gamma'], reward_type=1)
    m_ana, e_ana = evaluate_policy(ana_fn,
                                   kappa=ps['kappa'], sigma=ps['sigma'],
                                   lam=ps['lam'], gamma=ps['gamma'], reward_type=1)
    sens_rows.append({'参数组合': ps['label'],
                      'PPO 均值': round(m_rl, 4),  'PPO ±2SE': round(2*e_rl, 4),
                      '解析最优': round(m_ana, 4)})

df_sens = pd.DataFrame(sens_rows)
print(df_sens.to_string(index=False))

# 柱状图对比
x = np.arange(len(df_sens))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, df_sens['PPO 均值'],  0.4, label='PPO Oracle', color='steelblue')
ax.bar(x + 0.2, df_sens['解析最优'],  0.4, label='解析最优',   color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(df_sens['参数组合'], rotation=15, ha='right')
ax.set(ylabel='平均折扣回报', title='参数敏感性分析 — 奖励函数 (1)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_sensitivity.png', bbox_inches='tight')
plt.show()

## 11. 消融实验——学习率

固定其他所有超参数，仅改变学习率，在**奖励函数 (1) Oracle 模式**下训练 PPO。  
报告每个学习率的训练曲线与最终性能。

In [ ]:
lr_values  = [1e-4, 3e-4, 1e-3, 3e-3]
lr_results = {}

for lr in lr_values:
    print(f'  学习率 = {lr}')
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    actor_lr, log_lr = train_ppo(
        reward_type=1, oracle=True, n_steps=120_000,
        lr=lr, verbose=False
    )
    final_m, _ = evaluate_policy(ppo_policy(actor_lr), reward_type=1)
    lr_results[lr] = {'log': log_lr, 'final': final_m}
    print(f'    最终平均回报：{final_m:.4f}')

# 训练曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(lr_values)))
for (lr, res), col in zip(lr_results.items(), colors):
    steps, returns = res['log']
    ax1.plot(steps, returns, label=f'lr={lr}', color=col, lw=1.6)

ax1.axhline(mean_ana, color='k', ls='--', lw=1.2, label='解析最优')
ax1.set(title='学习率消融——训练曲线',
        xlabel='环境步数', ylabel='平均回合回报')
ax1.legend(fontsize=8)

finals = [lr_results[lr]['final'] for lr in lr_values]
ax2.bar([str(lr) for lr in lr_values], finals,
        color=[col for col in colors])
ax2.axhline(mean_ana, color='k', ls='--', lw=1.2, label='解析最优')
ax2.set(title='最终平均折扣回报', xlabel='学习率', ylabel='平均回报')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_ablation_lr.png', bbox_inches='tight')
plt.show()

print('\n汇总：')
for lr in lr_values:
    print(f'  lr={lr:<8}  最终回报={lr_results[lr]["final"]:.4f}')